In [12]:
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
from zipfile import ZipFile

from preflibtools.instances import OrdinalInstance

# PrefLib dataset zips matching your requested sources
PREFLIB_ZIPS = {
    "apa_election": "https://github.com/PrefLib/PrefLib-Data/releases/download/v1.0/00028_apa.zip",
    "sports_power_rankings": "https://github.com/PrefLib/PrefLib-Data/releases/download/v1.0/00056_seasonsport.zip",
    "eurovision": "https://github.com/PrefLib/PrefLib-Data/releases/download/v1.0/00064_eurovision.zip",
}

ORDINAL_EXTS = (".soc", ".soi", ".toc", ".toi")

def list_ordinal_files_in_zip(zip_url, limit=20):
    """Return ordinal-file members from a PrefLib zip (preview only)."""
    with TemporaryDirectory() as td:
        zip_path = Path(td) / "dataset.zip"
        urlretrieve(zip_url, zip_path)
        with ZipFile(zip_path, "r") as zf:
            members = [m for m in zf.namelist() if m.lower().endswith(ORDINAL_EXTS)]
    members.sort()
    return members[:limit], len(members)

def load_ordinal_instance_from_zip(zip_url, member_name=None):
    """Load one OrdinalInstance from a PrefLib dataset zip.

    If member_name is None, the first available ordinal file is used.
    """
    with TemporaryDirectory() as td:
        td_path = Path(td)
        zip_path = td_path / "dataset.zip"
        urlretrieve(zip_url, zip_path)

        with ZipFile(zip_path, "r") as zf:
            members = [m for m in zf.namelist() if m.lower().endswith(ORDINAL_EXTS)]
            members.sort()
            if not members:
                raise ValueError("No ordinal files found in zip archive.")
            target = member_name if member_name is not None else members[0]
            if target not in members:
                raise ValueError(f"Requested member not found: {target}")
            zf.extract(target, td_path)

        local_file = td_path / target
        instance = OrdinalInstance()
        instance.parse_file(str(local_file))
        return instance, target

# Preview what is inside each dataset zip
for name, url in PREFLIB_ZIPS.items():
    preview, total = list_ordinal_files_in_zip(url, limit=8)
    print(f"\n{name}: {total} ordinal files")
    for p in preview:
        print("  ", p)

# Example: load one APA election file (keeps downstream cells using `instance` working)
instance, loaded_member = load_ordinal_instance_from_zip(PREFLIB_ZIPS["apa_election"])
# Handle API differences across preflibtools versions
num_candidates = getattr(instance, "num_candidates", getattr(instance, "num_alternatives", None))
if num_candidates is None and hasattr(instance, "alternatives_name"):
    num_candidates = len(instance.alternatives_name)




apa_election: 24 ordinal files
   00028-00000001.soi
   00028-00000001.toc
   00028-00000002.soi
   00028-00000002.toc
   00028-00000003.soi
   00028-00000003.toc
   00028-00000004.soi
   00028-00000004.toc

sports_power_rankings: 4981 ordinal files
   00056-00000001.soi
   00056-00000002.soc
   00056-00000003.soi
   00056-00000004.soc
   00056-00000005.soc
   00056-00000006.soc
   00056-00000007.soc
   00056-00000008.soi

eurovision: 73 ordinal files
   00064-00000001.soi
   00064-00000002.soi
   00064-00000003.soi
   00064-00000004.soi
   00064-00000005.soi
   00064-00000006.soi
   00064-00000007.soi
   00064-00000008.soi


In [13]:
# Load one file from each dataset
instance_apa, _ = load_ordinal_instance_from_zip(PREFLIB_ZIPS["apa_election"])
instance_sports, _ = load_ordinal_instance_from_zip(PREFLIB_ZIPS["sports_power_rankings"])
instance_eurovision, _ = load_ordinal_instance_from_zip(PREFLIB_ZIPS["eurovision"])

for label, inst in [("apa_election", instance_apa), ("sports_power_rankings", instance_sports), ("eurovision", instance_eurovision)]:
    n_c = getattr(inst, "num_candidates", getattr(inst, "num_alternatives", None))
    if n_c is None and hasattr(inst, "alternatives_name"):
        n_c = len(inst.alternatives_name)
    print(f"{label}: {inst.num_voters} voters, {n_c} candidates")

apa_election: 18723 voters, 5 candidates
sports_power_rankings: 5 voters, 31 candidates
eurovision: 19 voters, 19 candidates


In [14]:
import numpy as np

def iter_pref_orders_with_counts(pref_orders):
    """Yield (count, order) pairs across common preflibtools order containers."""
    if hasattr(pref_orders, "items"):
        for order, count in pref_orders.items():
            yield count, order
        return
    for entry in pref_orders:
        if isinstance(entry, tuple) and len(entry) == 2 and isinstance(entry[0], int):
            count, order = entry
            yield count, order
        else:
            yield 1, entry

def orders_to_rank_matrix(instance):
    m = getattr(instance, "num_candidates", getattr(instance, "num_alternatives", None))
    if m is None and hasattr(instance, "alternatives_name"):
        m = len(instance.alternatives_name)
    if m is None:
        raise ValueError("Could not infer number of candidates from instance.")

    matrix = []
    for count, order in iter_pref_orders_with_counts(instance.orders):
        rank_vector = [0] * (m + 1)  # 1-indexed items
        for rank, item in enumerate(order, start=1):
            if isinstance(item, tuple):  # handle ties
                for i in item:
                    rank_vector[i] = rank
            else:
                rank_vector[item] = rank
        for _ in range(count):
            matrix.append(rank_vector[1:])  # drop the 0th index
    return np.array(matrix)

R_apa         = orders_to_rank_matrix(instance_apa)
R_sports      = orders_to_rank_matrix(instance_sports)
R_eurovision  = orders_to_rank_matrix(instance_eurovision)

print("R_apa shape:        ", R_apa.shape,        "  (assessors x candidates)")
print("R_sports shape:     ", R_sports.shape,     "  (assessors x candidates)")
print("R_eurovision shape: ", R_eurovision.shape, "  (assessors x candidates)")

R_apa shape:         (292, 5)   (assessors x candidates)
R_sports shape:      (5, 31)   (assessors x candidates)
R_eurovision shape:  (19, 19)   (assessors x candidates)


In [16]:
R_apa[:10]

array([[0, 0, 1, 0, 0],
       [0, 0, 0, 0, 1],
       [1, 0, 0, 0, 0],
       [0, 1, 0, 0, 0],
       [0, 0, 2, 0, 1],
       [2, 3, 1, 4, 5],
       [3, 2, 1, 4, 5],
       [0, 2, 1, 0, 0],
       [4, 2, 1, 3, 5],
       [4, 3, 1, 2, 5]])